#  Feedback Aspect-Based Sentiment Analysis with Explainable AI of App Reviews


## Explainable AI Section


### Utilities

In [1]:
MODEL_DIR = "/content/distilroberta-model"
PREDS_CSV = "/content/review_preds.csv"

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().to("cuda" if torch.cuda.is_available() else "cpu")

#### Stop-Word filter Customization

[The Full List of Negation and
Intensity words](https://aclanthology.org/attachments/P17-1154.Notes.pdf)

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")
stopwords = nlp.Defaults.stop_words
#print(stopwords)

# Removal of negation words
stopwords -= {"no", "not", "none", "never", "neither", "nobody",
"nothing", "nowhere", "seldom", "scarcely",
"hardly", "barely", "is not", "cannot", "may not",
"could not", "would not", "did not", "do not",
"does not", "was not", "are not", "were not"}

# Removal of intensity words
stopwords -= {"awfully", "extraordinary", "unusual", "much",
"rather", "very", "entirely", "greatly", "really",
"exceedingly", "too", "completely", "terribly",
"perfectly", "quite", "certainly", "especially",
"extremely", "fairly", "highly", "increasingly",
"much more", "particularly", "probably",
"more", "absolutely", "intensely", "supremely",
"most", "pretty"}

In [3]:
import re, string, numpy as np
from typing import List, Tuple, Optional, Dict
import torch


def clean_span(text: str) -> str:
  """Trim punctuation and whitespace around extracted phrase"""
  return re.sub(r"^\W+|\W+$", "", text.strip())


def is_informative(token: str) -> bool:
    """Filter tokens using spaCy stopwords + punctuation."""
    t = token.lower().strip()
    if not t:
        return False
    if t in nlp.Defaults.stop_words:
        return False
    if all(ch in string.punctuation for ch in t):
        return False
    return True

def phrases_from_offsets(text: str, seq_ids: List[Optional[int]],
                         offsets: List[Tuple[int,int]],
                         scores: np.ndarray,
                         which_seq: int = 0,
                         topk: int = 6) -> List[str]:
  "Merge top-salient tokens (by scores) into readable phrases using offsets"
  idxs = [j for j,(sid,off) in enumerate(zip(seq_ids, offsets)) if sid==which_seq and off and off[1]>off[0]]
  if not idxs:
    return []
  svals = np.array([scores[j] for j in idxs])
  order = np.argsort(-svals)
  top_idxs = [idxs[i] for i in order[:max(topk,1)]]
  top_idxs.sort()

   #merge adjacent/nearby tokens
  spans, cur = [], None
  for j in top_idxs:
    start, end = offsets[j]
    if cur is None: cur = [start,end]
    elif start <= cur[1] + 1: cur[1] = max(cur[1], end)
    else: spans.append(tuple(cur)); cur = [start,end]
  if cur: spans.append(tuple(cur))

  out = []
  seen = set()
  for s,e in spans:
    frag = clean_span(text[s:e])
    if is_informative(frag):
        key = frag.lower()
        if key not in seen:
          seen.add(key); out.append(frag)
  # backfill in case everything filtered
  if not out:
    for j in top_idxs:
      s,e = offsets[j]
      frag = clean_span(text[s:e])
      if frag: out.append(frag)
      if len(out) >= topk: break
  return out[:topk]


### Integrated Gradients Captum

In [4]:
!pip install -q captum

In [5]:
from captum.attr import IntegratedGradients

def _forward_probs(mdl, input_ids, attention_mask):
    outputs = mdl(input_ids=input_ids, attention_mask=attention_mask)
    return torch.softmax(outputs.logits, dim=-1)

def ig_token_importances(mdl, tok, texts: List[str], aspects: List[Optional[str]], max_length=256, target=None):
    """
    Returns per-token IG scores aligned to tokenizer offsets (fast tokenizer required).
    """
    device = next(mdl.parameters()).device
    encoded = tok(
        texts,
        text_pair=aspects if any(aspects) else None,
        padding=True, truncation=True, max_length=max_length,
        return_tensors="pt", return_offsets_mapping=True
    )
    offsets = encoded.pop("offset_mapping")   # (B,T,2)
    encs = encoded.encodings
    encoded = {k:v.to(device) for k,v in encoded.items()}

    # Choose target per-example (model's argmax) if not provided
    with torch.no_grad():
        probs = _forward_probs(mdl, encoded["input_ids"], encoded["attention_mask"])
        targets = probs.argmax(dim=-1) if target is None else torch.tensor([target]*probs.size(0), device=device)

    ig = IntegratedGradients(lambda ids, attn: _forward_probs(mdl, ids, attn).gather(1, targets.view(-1,1)).squeeze(1))

    # baselines: pad tokens (ids=tok.pad_token_id), same attention mask
    baselines = torch.full_like(encoded["input_ids"], fill_value=tok.pad_token_id)
    atts = encoded["attention_mask"]

    attributions = ig.attribute(inputs=encoded["input_ids"],
                                baselines=baselines,
                                additional_forward_args=(atts,),
                                n_steps=32)  # trade-off speed/quality
    # aggregate embedding dims (already token ids, so attribute per token index)
    token_scores = attributions.abs().sum(dim=-1).detach().cpu().numpy()  # (B,T)

    return token_scores, probs.detach().cpu().numpy(), encs, offsets.cpu().tolist()

#### Demo


In [7]:
import os, re, string, math, numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from typing import List, Optional, Tuple, Dict, Any
from captum.attr import IntegratedGradients
import spacy
from IPython.display import HTML, display

# -------------------------------CONFIG-------------------------------
CSV_PATH = '/content/review_preds.csv'
SAMPLE_N = 30          # how many rows to visualize (set None for all)
MAX_LEN  = 128         # tokenizer max_length
BATCH    = 2          # attribution batch size
IG_STEPS = 12          # IG steps,
USE_ASPECT_PAIR = True # pair text & aspect (recommended for ABSA)
ASPECT_COL_FALLBACK = None

# -------------------------------CHECKS-------------------------------

assert 'model' in globals() and 'tokenizer' in globals(), "Run: model, tokenizer = load_model_and_tokenizer() first."
device = next(model.parameters()).device
model.to(device).eval()
print("Model device:", device)

# ---------- spaCy stopwords ----------
nlp = spacy.load("en_core_web_sm")
def clean_span(s): return re.sub(r"^\W+|\W+$", "", s.strip())
def is_stopword(tok):
    t = tok.lower().strip()
    return (not t) or (t in nlp.Defaults.stop_words) or all(ch in string.punctuation for ch in t)

# ---------- CSV load + cols ----------
def _first(path_list):
    for p in path_list:
        if os.path.exists(p): return p
    raise FileNotFoundError(path_list)

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH}")

text_candidates   = ["text","review","content","review_text","sentence","clean_text","comment","body"]
aspect_candidates = ["aspect","category","target","feature","entity","aspect_term","tag"]

text_col   = next((c for c in text_candidates if c in df.columns), None)
aspect_col = next((c for c in aspect_candidates if c in df.columns), ASPECT_COL_FALLBACK)
if not text_col: raise ValueError(f"No text column found. Columns: {list(df.columns)}")
print(f"Using text column: {text_col}")
print("Using aspect column:", aspect_col if aspect_col and USE_ASPECT_PAIR else "None")

vis_df = df if SAMPLE_N is None else df.sample(min(SAMPLE_N, len(df)), random_state=42).reset_index(drop=True)

texts   = vis_df[text_col].astype(str).tolist()
aspects = (vis_df[aspect_col].astype(str).tolist() if (aspect_col and USE_ASPECT_PAIR) else [None]*len(texts))
# treat string 'nan' / NaN as None
aspects = [None if (a is None or str(a).lower() == "nan") else a for a in aspects]

true_label_col = next((c for c in ["true_label","gold","y_true","y","label"] if c in vis_df.columns), None)

# ---------- tokenizer prechecks ----------
if not getattr(tokenizer, "is_fast", False):
    raise RuntimeError("This visualization requires a *fast* tokenizer (use_fast=True). Reload your tokenizer with use_fast=True.")

# ---------- IG over inputs_embeds ----------
emb_layer = model.get_input_embeddings()
ig = IntegratedGradients(lambda inputs_embeds, attention_mask:
                         model(inputs_embeds=inputs_embeds,
                               attention_mask=attention_mask,
                               return_dict=True).logits)

def ig_batch_attributions(t_list: List[str], a_list: List[Optional[str]]):
    """Returns (token_scores, probs, preds, offsets_list, encs). Never returns None."""
    try:
        enc = tokenizer(
            t_list,
            text_pair=a_list if any(a_list) else None,
            padding=True, truncation=True, max_length=MAX_LEN,
            return_tensors="pt", return_offsets_mapping=True
        )
        if not hasattr(enc, "encodings") or enc.encodings is None:
            raise RuntimeError("tokenizer returned no fast encodings; ensure use_fast=True.")

        offsets = enc.pop("offset_mapping")     # (B,T,2) torch tensor
        encs    = enc.encodings

        input_ids      = enc["input_ids"].to(device, non_blocking=True)
        attention_mask = enc["attention_mask"].to(device, non_blocking=True)

        with torch.no_grad():
            logits = model(input_ids=input_ids, attention_mask=attention_mask, return_dict=True).logits
            probs  = torch.softmax(logits, dim=-1)
            preds  = probs.argmax(dim=-1)

        # embeddings & baselines
        input_embeds = emb_layer(input_ids)
        pad_id = tokenizer.pad_token_id
        if pad_id is None:  # fallback for models without pad token set
            pad_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0
        baseline_ids    = torch.full_like(input_ids, pad_id)
        baseline_embeds = emb_layer(baseline_ids)

        atts = ig.attribute(
            inputs=input_embeds,
            baselines=baseline_embeds,
            additional_forward_args=(attention_mask,),
            target=preds,
            n_steps=IG_STEPS
        )
        token_scores = atts.sum(dim=-1).detach().cpu().numpy()

        out = (token_scores,
               probs.detach().cpu().numpy(),
               preds.detach().cpu().numpy(),
               offsets.detach().cpu().tolist(),
               encs)
        return out
    except Exception as e:
        # surface the reason explicitly
        raise RuntimeError(f"ig_batch_attributions failed on batch of size {len(t_list)}: {type(e).__name__}: {e}")

def color_token_html(tok: str, score: float, max_abs: float) -> str:
    if max_abs <= 1e-12: return tok
    a = min(1.0, abs(score)/max_abs); a = math.tanh(2.0*a)
    bg = ("rgba(0,200,0,{a})".format(a=a) if score >= 0 else "rgba(200,0,0,{a})".format(a=a))
    return f'<span style="background:{bg}; padding:0 2px; border-radius:3px;">{tok}</span>'

rows = []
with tqdm(total=len(texts), desc="Attributing", ncols=90) as pbar:
    for i0 in range(0, len(texts), BATCH):
        b_texts   = texts[i0:i0+BATCH]
        b_aspects = aspects[i0:i0+BATCH]

        # --- this will now raise with an explicit message instead of returning None
        scores, probs, preds, offs, encs = ig_batch_attributions(b_texts, b_aspects)

        for bi, (text, aspect) in enumerate(zip(b_texts, b_aspects)):
            prob_vec = probs[bi]
            pred_idx = int(preds[bi])
            label_map = (id2label if 'id2label' in globals() and isinstance(id2label, dict) else {0:"neg",1:"neu",2:"pos"})
            label_name = label_map.get(pred_idx, str(pred_idx))

            seq_ids = encs[bi].sequence_ids
            offsets = offs[bi]
            idxs = [j for j,(sid,o) in enumerate(zip(seq_ids, offsets)) if sid==0 and o and o[1]>o[0]]
            svals = np.array([scores[bi][j] for j in idxs]) if idxs else np.array([])
            max_abs = float(np.max(np.abs(svals))) if svals.size else 1.0

            pretty = []
            for j in idxs:
                s,e = offsets[j]
                frag = clean_span(text[s:e])
                if not frag: continue
                if frag.lower() in {"not","no","never"} or not is_stopword(frag):
                    pretty.append(color_token_html(frag, scores[bi][j], max_abs))
                else:
                    pretty.append(frag)

            word_html = " ".join(pretty) if pretty else text
            true_lbl = vis_df.iloc[i0+bi][true_label_col] if true_label_col else ""
            rows.append({
                "True Label": true_lbl,
                "Predicted Label": f"{label_name} ({prob_vec[pred_idx]:.2f})",
                "Attribution Label": label_name,
                "Attribution Score": f"{np.sum(svals):.2f}" if svals.size else "0.00",
                "Word Importance": word_html
            })

        pbar.update(len(b_texts))

viz = pd.DataFrame(rows)
styles = [
    dict(selector="th", props=[("text-align","left"),("padding","6px")]),
    dict(selector="td", props=[("text-align","left"),("padding","6px"),("vertical-align","top")]),
    dict(selector="tr:nth-child(even)", props=[("background","#f7f7f7")]),
]

sty = viz.style.set_table_styles(styles)
try:
    sty = sty.hide(axis="index")
except Exception:
    viz = viz.reset_index(drop=True) # fallback: drop old index
    sty = viz.style.set_table_styles(styles)

html = sty.to_html(escape=False)
display(HTML("<h3>IMDB-style Word Importance (Captum IG, robust)</h3>" + html))

Model device: cuda:0
Loaded 972 rows from /content/review_preds.csv
Using text column: text
Using aspect column: aspect


Attributing:   0%|                                                 | 0/30 [00:00<?, ?it/s]